In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Wed Mar  4 21:30:30 2026

@author: Mehdi Ayouz
"""

"""
CODE 1 â€” Hamiltonien qubit pour lâ€™atome H (1 Ã©lectron)
Base STO-3G
Distance interne pas nÃ©cessaire (un seul noyau)
"""

import numpy as np

# ==========================================================
# ETAPE 1 â€” Hamiltonien Ã©lectronique
# ==========================================================

"""
Hamiltonien Ã©lectronique de H (Born-Oppenheimer) :

H = - (âˆ‡^2)/2 - Z / r
Avec Z=1

Dans une base Ï†_0 (1s STO-3G), lâ€™hamiltonien devient :

H = h_00 a_0â€  a_0

avec
h_00 = âˆ« Ï†_0*(r) (-âˆ‡^2/2 - 1/r) Ï†_0(r) dr

Pas de terme 2 Ã©lectrons car 1 seul Ã©lectron.
"""
# ==========================================================
# 1. Orbital simple H (STO-3G simplifiÃ©)
# ==========================================================
# phi(r) = (2*alpha/pi)^(3/4) * exp(-alpha*r^2)
alpha = 0.5

# ==========================================================
# 1. Orbital H 1s STO-3G complet
# ==========================================================
# STO-3G coefficients pour H 1s
d = np.array([0.444635, 0.535328, 0.154329])
alpha_k = np.array([0.109818, 0.405771, 2.22766])

def phi_raw(r):
    """
    Orbital H 1s STO-3G non normalisÃ©e : sum_k d_k * exp(-alpha_k * r^2)
    """
    phi_r = np.zeros(len(r))
    for i in range(len(r)):
        phi_r[i] = np.dot(d, np.exp(-alpha_k * r[i]**2))
    return phi_r

# ==========================================================
# 2. Normalisation radiale de phi
# ==========================================================
def normalize_phi(phi_r, r):
    """
    Normalise phi(r) pour une fonction spherique
    âˆ« |phi(r)|^2 4 pi r^2 dr = 1
    """
    dr = r[1] - r[0]
    norm = np.sum(np.abs(phi_r)**2 * 4*np.pi * r**2) * dr
    factor = 1.0 / np.sqrt(norm)
    phi_normalized_r = phi_r * factor
    return phi_normalized_r, factor

# ==========================================================
# 3. Laplacian radial
# âˆ‡Â² phi(r) = dÂ²phi/drÂ² + 2/r dphi/dr
# ==========================================================
def laplacian_phi_raw(r):
    """
    Laplacien radial pour orbital STO-3G non normalisÃ©
    """
    dphi_dr = np.zeros(len(r))
    d2phi_dr2 = np.zeros(len(r))
    
    for i in range(len(r)):
        dphi_dr[i] = np.sum(-2*alpha_k*d*r[i]*np.exp(-alpha_k*r[i]**2))
        d2phi_dr2[i] = np.sum((4*alpha_k**2*r[i]**2 - 2*alpha_k)*d*np.exp(-alpha_k*r[i]**2))
    
    laplacian = d2phi_dr2 + 2/r*dphi_dr
    return laplacian

# ==========================================================
# 4. Calcul de h_00 = <phi| -âˆ‡Â²/2 - 1/r |phi>
# ==========================================================
def h_00_integral(rmax=20.0, N=10000):
    r = np.linspace(1e-6, rmax, N)
    dr = r[1] - r[0]
    
    # phi non normalisÃ©
    phi_r = phi_raw(r)
    lap_phi = laplacian_phi_raw(r)
    
    # Normalisation
    phi_r, factor = normalize_phi(phi_r, r)
    lap_phi *= factor  # Laplacian doit Ãªtre mis Ã  lâ€™Ã©chelle par le mÃªme facteur
    
    # IntÃ©grande : phi(r) * (-1/2 laplacian - 1/r) * r^2 * 4Ï€
    integrand = phi_r * (-0.5 * lap_phi - phi_r / r) * r**2 * 4*np.pi
    h00 = np.sum(integrand) * dr
    return h00

# ==========================================================
# 5. Transformation Jordan-Wigner pour 1 qubit
# ==========================================================
# Pour 1 electron et 1 qubit : 
# a0â€  a0 = (I - Z)/2
# Donc H_qubit = g0*I + g1*Z avec :
# g0 = h00/2
# g1 = -h00/2

def hamiltonian_qubit(h00):
    g0 = h00 / 2
    g1 = - h00 / 2
    
    # Matrices Pauli 2x2
    I = np.array([[1,0],[0,1]])
    Z = np.array([[1,0],[0,-1]])
    
    H_qubit = g0 * I + g1 * Z
    return H_qubit, g0, g1

# ==========================================================
# 6. ExÃ©cution
# ==========================================================
h_00 = h_00_integral()
H_qubit, g0, g1 = hamiltonian_qubit(h_00)

print("h_00 =", h_00)
print("g0 =", g0)
print("g1 =", g1)
print("Hamiltonien qubit 2x2 :\n", H_qubit)

# ==========================================================
# 2. IntÃ©grale 1 Ã©lectron
# ==========================================================
# h_00 = <phi| -âˆ‡^2/2 - 1/r |phi>
# Valeur analytique connue pour gaussienne simple H 1s
#h_00 = -0.5  # a.u.

# ==========================================================
# ETAPE 3 â€” Pauli et Jordan-Wigner
# ==========================================================

"""
Pour un seul qubit :

a_0â€  a_0 = (I - Z)/2

- Cette formule exprime le nombre d'electrons en termes
  de matrices de Pauli pour 1 qubit.
- Elle est standard et dÃ©coule des propriÃ©tÃ©s des matrices
  de Pauli (XÂ² = YÂ² = ZÂ² = I, XY = iZ, ...).

DÃ©finition des opÃ©rateurs fermioniques pour 1 qubit :

a_0     = (X + i Y) / 2
a_0â€     = (X - i Y) / 2

Hamiltonien qubit :

On remplace a_0â€  a_0 dans H :

H = h_00 * a_0â€  a_0
  = h_00 * (1/2) * (I - Z)
  = (h_00 / 2) * I - (h_00 / 2) * Z
  
Donc Hamiltonien qubit :

H_qubit = g0 * I + g1 * Z
"""

I_mat = np.array([[1,0],[0,1]])
Z_mat = np.array([[1,0],[0,-1]])

g0 = h_00/2
g1 = -h_00/2

H_qubit = g0 * I_mat + g1 * Z_mat

# ==========================================================
# ETAPE 4 â€” Affichage
# ==========================================================

print("Hamiltonien qubit H (1 Ã©lectron) :")
print(H_qubit)

print("\nCoefficients g_i :")
print("g0 =", g0)
print("g1 =", g1)


"""
CODE 2 â€” VQE pour lâ€™atome H (1 Ã©lectron)
"""

import numpy as np
import math

# ==========================================================
# Hamiltonien qubit H
# ==========================================================

# MÃªme g0/g1 que CODE 1
g0 = -0.25
g1 = 0.25

I_mat = np.array([[1,0],[0,1]])
Z_mat = np.array([[1,0],[0,-1]])

H_qubit = g0*I_mat + g1*Z_mat

# ==========================================================
# Ansatz variationnel
# ==========================================================

"""
Pour 1 Ã©lectron / 1 qubit :

|Ïˆ(Î¸)> = cos(Î¸)|0> + sin(Î¸)|1>

Mais H est diagonal â†’ lâ€™Ã©tat fondamental est |1>
"""

def psi(theta):
    return np.array([math.cos(theta), math.sin(theta)])

# ==========================================================
# Fonction Ã©nergie
# ==========================================================

def energy(theta):
    state = psi(theta)
    return np.real(np.conjugate(state) @ H_qubit @ state)

# ==========================================================
# Optimisation brute
# ==========================================================

theta_vals = np.linspace(0, math.pi/2, 100)
best_E = 1e10
best_theta = 0

for theta in theta_vals:
    E = energy(theta)
    if E < best_E:
        best_E = E
        best_theta = theta

print("\nTheta optimal :", best_theta)
print("Energie VQE :", best_E)

h_00 = -0.44869664519902597
g0 = -0.22434832259951298
g1 = 0.22434832259951298
Hamiltonien qubit 2x2 :
 [[ 0.          0.        ]
 [ 0.         -0.44869665]]
Hamiltonien qubit H (1 Ã©lectron) :
[[ 0.          0.        ]
 [ 0.         -0.44869665]]

Coefficients g_i :
g0 = -0.22434832259951298
g1 = 0.22434832259951298

Theta optimal : 1.5707963267948966
Energie VQE : -0.5
